# Coding Homework 05: Eigenvalues, the Spectral Theorem, and a Complete Forward Block

This assignment connects eigenvalue geometry to the weight matrices in your cumulative mini-transformer. You will implement two spectral algorithms, measure which directions a matrix uses, and complete one forward transformer block with attention, a feed-forward map, and residual additions.

**Estimated time:** 2–3 hours
**Submit:** this notebook and `mini_transformer.py`


## 0. Setup and cumulative-model check

Keep `mini_transformer.py` beside this notebook. The completed HW03 attention and HW04 orthogonal-initialization functions are already supplied. Your new HW05 model work belongs in the same student-owned file; no instructor utility module is needed.


In [1]:
import numpy as np
import mini_transformer as mt

np.set_printoptions(precision=5, suppress=True)
W_check = mt.orthogonal_matrix(4, 4, seed=412)
assert np.allclose(W_check.T @ W_check, np.eye(4), atol=1e-10)
print("Setup complete; the cumulative HW04 model is available.")


Setup complete; the cumulative HW04 model is available.


## 1. Power iteration: repeated action reveals a dominant direction

For a real symmetric matrix with one eigenvalue whose absolute value is strictly largest, repeated multiplication amplifies that eigendirection relative to the others. Starting with a unit vector $x$, repeat

$$y=Ax, \qquad x_{\text{new}}=y/\lVert y\rVert.$$

After normalization, estimate the eigenvalue with the **Rayleigh quotient** $\lambda=x^TAx$. This formula preserves the sign; `norm(A @ x)` does not. Flip `x_new` when its dot product with the old `x` is negative so the vector does not alternate signs. Stop when the residual $\lVert Ax-\lambda x\rVert$ is at most `tol`.


In [2]:
def power_iteration(A, n_iter=100, tol=1e-10, seed=0):
    """Return (dominant eigenvalue, unit eigenvector, estimate history).

    A must be a real symmetric square matrix with a unique dominant
    eigenvalue in absolute value.
    """
    # Validate that A is symmetric and square
    A = np.asarray(A, dtype=float)

    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("Matrix A must be square.")

    if not np.allclose(A, A.T, atol=1e-8):
        raise ValueError("Matrix A must be symmetric.")

    if not isinstance(n_iter, (int, np.integer)):
        raise TypeError("n_iter must be an integer.")

    if n_iter <= 0:
        raise ValueError("n_iter must be positive.")

    if tol <= 0:
        raise ValueError("tol must be positive.")

    n = A.shape[0]
    # Create a reproducible unit starting vector
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(n)
    x = x / np.linalg.norm(x)

    history = []
    for _ in range(n_iter):
        y = A @ x
        # Rayleigh quotient estimate for eigenvalue
        lam = x.T @ y
        history.append(lam)

        # Check for convergence using the residual norm ||Ax - lam*x||
        residual = y - lam * x
        if np.linalg.norm(residual) <= tol:
            break

        # Normalize the vector for the next iteration
        x_new = y / np.linalg.norm(y)

        # Flip direction if dot product with old vector is negative to avoid alternating signs
        if x_new @ x < 0:
            x_new = -x_new

        x = x_new

    return lam, x, history


In [3]:
A_power = np.array([[4., 1.], [1., 2.]])
lam, v, history = power_iteration(A_power, seed=412)
assert np.isclose(np.linalg.norm(v), 1.0)
assert np.allclose(A_power @ v, lam * v, atol=1e-8)
assert np.isclose(lam, np.linalg.eigvalsh(A_power)[-1], atol=1e-8)

lam_negative, v_negative, _ = power_iteration(np.diag([-5., 2.]), seed=9)
assert np.isclose(lam_negative, -5.0, atol=1e-8)
assert np.allclose(np.diag([-5., 2.]) @ v_negative, lam_negative * v_negative, atol=1e-8)
print(f"Dominant eigenvalue: {lam:.8f}; iterations: {len(history)}")


Dominant eigenvalue: 4.41421356; iterations: 24


## 2. Spectral theorem: orthogonal directions reconstruct a symmetric matrix

The spectral theorem says that a real symmetric matrix can be written

$$A=Q\Lambda Q^T=\sum_i \lambda_i q_iq_i^T,$$

where the columns of $Q$ are orthonormal eigenvectors. NumPy's `np.linalg.eigh` returns eigenvalues in ascending order and eigenvectors in columns. Complete the function so both are reordered from largest eigenvalue to smallest. Reject nonsymmetric input rather than applying a symmetric-matrix theorem where it does not belong.


In [4]:
def sorted_spectral_decomposition(A, tol=1e-10):
    """Return eigenvalues and orthonormal eigenvectors in descending order."""
    A = np.asarray(A, dtype=float)

    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("Matrix A must be square.")

    if not np.allclose(A, A.T, atol=tol):
        raise ValueError("Matrix A must be symmetric.")

    evals, Q = np.linalg.eigh(A)

    idx = np.argsort(evals)[::-1]

    return evals[idx], Q[:, idx]


In [5]:
A_spectral = np.array([[4., 2.], [2., 3.]])
evals, Q = sorted_spectral_decomposition(A_spectral)
assert np.all(np.diff(evals) <= 0)
assert np.allclose(Q.T @ Q, np.eye(2), atol=1e-10)
assert np.allclose(Q @ np.diag(evals) @ Q.T, A_spectral, atol=1e-10)
print("Eigenvalues (descending):", evals)


Eigenvalues (descending): [5.56155 1.43845]


## 3. Gram spectra and effective rank

For any matrix $W$, the Gram matrix $G=W^TW$ is symmetric positive semidefinite. Its eigenvalues are the squared stretch factors of $W$: a zero eigenvalue marks a direction erased by the map, while a very small one marks a direction used only weakly.

Define the effective rank to be the number of eigenvalues of $W^TW$ larger than `threshold_frac` times the largest eigenvalue. Return `0` for a zero matrix. This is a simple threshold-based summary, not the exact algebraic rank.


In [6]:
def effective_rank(W, threshold_frac=0.01):
    """Count Gram eigenvalues above threshold_frac times the largest."""

    W = np.asarray(W, dtype=float)

    if W.ndim != 2:
        raise ValueError("W must be a 2-D matrix.")

    if not isinstance(threshold_frac, (int, float, np.integer, np.floating)):
        raise TypeError("threshold_frac must be numeric.")

    if threshold_frac < 0:
        raise ValueError("threshold_frac must be nonnegative.")

    if np.allclose(W, 0):
        return 0

    G = W.T @ W

    evals, _ = sorted_spectral_decomposition(G)

    largest_eval = evals[0]
    threshold = threshold_frac * largest_eval

    return int(np.sum(evals > threshold))


In [7]:
W_demo = np.diag([3., 1., 0.01])
assert effective_rank(W_demo, threshold_frac=0.05) == 2
assert effective_rank(np.zeros((4, 3))) == 0

W_Q_demo, _, _ = mt.orthogonal_projection_matrices(4, 4, seed=412)
gram_evals, _ = sorted_spectral_decomposition(W_Q_demo.T @ W_Q_demo)
assert np.allclose(gram_evals, np.ones(4), atol=1e-10)
print("Orthogonal W_Q Gram eigenvalues:", gram_evals)
print("Effective rank:", effective_rank(W_Q_demo))


Orthogonal W_Q Gram eigenvalues: [1. 1. 1. 1.]
Effective rank: 4


## 4. Add a feed-forward map to `mini_transformer.py`

Attention mixes information **between token rows**. A feed-forward map then transforms each row independently using the same two affine maps:

$$\operatorname{FFN}(X)=\operatorname{ReLU}(XW_1+b_1)W_2+b_2.$$

`ReLU` replaces each negative entry by zero. The conversion and shape-checking code is supplied in `mini_transformer.py`; implement only the displayed mathematics in `feed_forward`.


In [8]:
X_ff = np.array([[1., -2.], [0., 3.]])
W1_ff = np.array([[1., 0., -1.], [0., 2., 1.]])
b1_ff = np.array([0., 1., 0.])
W2_ff = np.array([[1., 0.], [0., 1.], [1., -1.]])
b2_ff = np.array([0.5, -0.5])
ff_out = mt.feed_forward(X_ff, W1_ff, b1_ff, W2_ff, b2_ff)
ff_expected = mt.relu(X_ff @ W1_ff + b1_ff) @ W2_ff + b2_ff
assert ff_out.shape == X_ff.shape
assert np.allclose(ff_out, ff_expected)
print("Feed-forward output:\n", ff_out)


Feed-forward output:
 [[ 1.5 -0.5]
 [ 3.5  3.5]]


## 5. Mini-transformer milestone: one complete forward block

A **residual connection** adds a sublayer's input back to its output. If `Attn(X)` and `FFN(H)` have the same shape as their inputs, this block is

$$H=X+\operatorname{Attn}(X), \qquad Y=H+\operatorname{FFN}(H).$$

This is not yet a trained language model. It is one small, transparent forward block built from the matrix operations you know. Implement `transformer_block` in `mini_transformer.py` by composing the existing attention head with the new feed-forward map and the two displayed additions.


In [9]:
X = np.array([[1., 0., 1., 0.],
              [0., 1., 1., 0.],
              [1., 1., 0., 1.]])
W_Q, W_K, W_V = mt.orthogonal_projection_matrices(4, 4, seed=412)
rng = np.random.default_rng(412)
W1 = rng.standard_normal((4, 7)) / np.sqrt(4)
W2 = rng.standard_normal((7, 4)) / np.sqrt(7)
b1 = np.zeros(7)
b2 = np.zeros(4)
mask = mt.causal_mask(len(X))

Y = mt.transformer_block(X, W_Q, W_K, W_V, W1, b1, W2, b2, mask=mask)
assert Y.shape == X.shape
assert np.all(np.isfinite(Y))
assert not np.shares_memory(Y, X)
print("Complete forward-block output:\n", Y)


Complete forward-block output:
 [[-0.26277 -1.13141  2.71608 -1.01187]
 [-0.4909  -0.79074  2.08922 -0.9072 ]
 [ 0.81622 -0.04019  1.60371  0.08858]]


### Short interpretation

In 2–4 sentences, explain what the eigenvalues of $W_Q^TW_Q$ say about directions preserved or suppressed by $W_Q$, and why an orthogonal initialization begins with a flat Gram spectrum.


In [10]:
spectral_interpretation = """
The eigenvalues tell us how strongly the weight matrix stretches different input directions.
Large eigenvalues correspond to directions that are emphasized, while small eigenvalues
correspond to directions that are weakened or nearly removed.
With orthogonal initialization, all directions start with the same strength,
so no direction is preferred at the beginning.
""".strip()
assert len(spectral_interpretation.split()) >= 25
print(spectral_interpretation)


The eigenvalues tell us how strongly the weight matrix stretches different input directions.
Large eigenvalues correspond to directions that are emphasized, while small eigenvalues
correspond to directions that are weakened or nearly removed.
With orthogonal initialization, all directions start with the same strength,
so no direction is preferred at the beginning.


## 6. Submission and AI-use note

Restart the kernel, run all cells, and confirm every assertion passes. Keep this notebook alongside `mt_utils.py` / `mini_transformer.py` in the same folder when re-running.



In [11]:
ai_use_note = "No AI assistance used."  # Replace if applicable
assert ai_use_note.strip()
print(ai_use_note)


No AI assistance used.
